# 08. 기준값 조정으로 재현율 높이기 (GPU 필요 없음, 약 1분)

07의 `정리모델_3개`는 정밀도가 높은 대신 재현율이 낮다(특히 구토·오심, 두통, 어지러움).
모델은 다시 학습하지 않고, **"확률이 몇 이상이면 있음"으로 볼지(기준값)**만 증상마다 바꿔 본다.

- 07에서 저장한 Val 확률(`07_결과/정리모델_3개_val_prob.npy`)만 읽는다. 모델 가중치는 건드리지 않는다.
- 기준은 **고친 Val**(Claude가 고친 Val).
- 기준값을 Val로 고르면 점수가 부풀려지므로 **Val을 반으로 나눈다**: A 절반으로 기준값을 고르고, B 절반으로 채점한다.
- 결과: `results/08_결과/` (기준값 표, 점수표).

**돌리는 법**: 런타임 유형은 CPU로 충분하다. 런타임 → 모두 실행.

## 0. 준비

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, glob
PROJECT_ROOT = "/content/drive/MyDrive/증상인식_최종_프로젝트"   # 폴더를 다른 곳에 두었으면 여기만 바꾸기
if not os.path.isdir(f"{PROJECT_ROOT}/src"):   # 못 찾으면 드라이브에서 자동으로 찾기
    found = [p for pat in ["/content/drive/**/증상인식_최종_프로젝트*", "/content/drive/.shortcut-targets-by-id/*/증상인식_최종_프로젝트*"]
             for p in glob.glob(pat, recursive=True) if os.path.isdir(f"{p}/src")]
    assert found, "프로젝트 폴더를 못 찾음. 공유 폴더를 '내 드라이브'에 바로가기로 추가했는지 확인"
    PROJECT_ROOT = found[0]
print("프로젝트 폴더:", PROJECT_ROOT)
os.environ["SYMPTOM_FINAL_ROOT"] = PROJECT_ROOT
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
from src import config
from src.data import load_labels
from src.evaluate import claude_fixed_val

OUT = f"{config.RESULTS_DIR}/08_결과"
os.makedirs(OUT, exist_ok=True)
LABELS = list(config.LABELS)
FIX = ["구토·오심", "두통", "어지러움"]   # 07에서 라벨을 고친 증상

_, va_df = load_labels()
fixed = claude_fixed_val(va_df)
assert fixed is not None, "고친 Val 파일(val_claude_judge.xlsx)이 없음"
Y, M = fixed
M = np.ones_like(Y) if M is None else M
probs = {"3개 수정 모델": np.load(f"{config.RESULTS_DIR}/07_결과/정리모델_3개_val_prob.npy"),
         "원본 모델": np.load(f"{config.RESULTS_DIR}/06_결과/원본모델_val_prob.npy")}
assert all(p.shape == Y.shape for p in probs.values()), "확률과 Val 크기가 다름"
p = probs["3개 수정 모델"]

# Val을 반으로: A(기준값 고르기) / B(채점)
idx = np.random.RandomState(42).permutation(len(Y))
A, B = np.sort(idx[: len(Y) // 2]), np.sort(idx[len(Y) // 2 :])
print(f"Val {len(Y):,}개 → A {len(A):,} / B {len(B):,}")

## 1. 도구: 기준값별 재현율·정밀도·F1
기준값을 증상마다 다르게 줄 수 있다(`thr`는 숫자 하나 또는 증상 8개 길이의 배열).

In [ ]:
GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)

def counts(Y, pred, M):
    tp = ((Y == 1) & (pred == 1) & (M == 1)).sum(0)
    fp = ((Y == 0) & (pred == 1) & (M == 1)).sum(0)
    fn = ((Y == 1) & (pred == 0) & (M == 1)).sum(0)
    return tp, fp, fn

def report(Y, prob, M, thr):
    """증상별 재현율·정밀도·F1 + Macro 행."""
    tp, fp, fn = counts(Y, (prob >= np.asarray(thr)).astype(int), M)
    t = pd.DataFrame({"재현율": tp / np.maximum(tp + fn, 1), "정밀도": tp / np.maximum(tp + fp, 1),
                      "F1": 2 * tp / np.maximum(2 * tp + fp + fn, 1)}, index=LABELS)
    t.loc["Macro"] = t.mean()
    return t.round(3)

def curve(Y, prob, M, j):
    """증상 j 하나의 기준값별 재현율·정밀도·F1."""
    rows = []
    for t in GRID:
        tp, fp, fn = counts(Y[:, [j]], (prob[:, [j]] >= t).astype(int), M[:, [j]])
        tp, fp, fn = tp[0], fp[0], fn[0]
        rows.append({"기준값": t, "재현율": tp / max(tp + fn, 1), "정밀도": tp / max(tp + fp, 1),
                     "F1": 2 * tp / max(2 * tp + fp + fn, 1)})
    return pd.DataFrame(rows).set_index("기준값")

## 2. A 절반으로 기준값 고르기
두 가지 규칙으로 고른다.
- **F1 최고**: F1이 가장 높은 기준값
- **재현율 우선**: 정밀도를 `MIN_PREC` 이상으로 지키면서 재현율이 가장 높은 기준값(조건을 맞추는 값이 없으면 0.5)

In [ ]:
MIN_PREC = 0.8

thr_f1, thr_rec = {}, {}
for j, s in enumerate(LABELS):
    c = curve(Y[A], p[A], M[A], j)
    thr_f1[s] = c["F1"].idxmax()
    ok = c[c["정밀도"] >= MIN_PREC]
    thr_rec[s] = ok["재현율"].idxmax() if len(ok) else 0.5

th = pd.DataFrame({"F1 최고": thr_f1, f"재현율 우선(정밀도≥{MIN_PREC})": thr_rec})
th.to_csv(f"{OUT}/기준값_A에서고름.csv", encoding="utf-8-sig")
display(th)

## 3. B 절반으로 채점
비교하는 설정
- 기준 0.5 (지금): 3개 수정 모델, 원본 모델
- F1 최고 기준값: 수정한 3개 증상만 / 8개 전부
- 재현율 우선 기준값: 수정한 3개 증상만

In [ ]:
def thr_vec(d, only=None):
    return np.array([d[s] if (only is None or s in only) else 0.5 for s in LABELS])

settings = {
    "원본 모델, 0.5": (probs["원본 모델"], 0.5),
    "3개 수정, 0.5 (지금)": (p, 0.5),
    "3개 수정, F1 최고(3개만)": (p, thr_vec(thr_f1, FIX)),
    "3개 수정, F1 최고(8개 전부)": (p, thr_vec(thr_f1)),
    "3개 수정, 재현율 우선(3개만)": (p, thr_vec(thr_rec, FIX)),
}
res = {k: report(Y[B], pr[B], M[B], t) for k, (pr, t) in settings.items()}

summary = pd.DataFrame({k: r.loc["Macro"] for k, r in res.items()}).T
summary.to_csv(f"{OUT}/점수_B절반_요약.csv", encoding="utf-8-sig")
print("[B 절반] Macro 평균 (재현율·정밀도·F1)"); display(summary)

detail = pd.concat({k: r for k, r in res.items()}, axis=1)
detail.to_csv(f"{OUT}/점수_B절반_증상별.csv", encoding="utf-8-sig")
print("[B 절반] 수정한 3개 증상"); display(detail.loc[FIX])

## 4. 참고: 수정한 3개 증상의 기준값별 곡선 (Val 전체)
기준값을 낮출수록 재현율이 오르고 정밀도가 내려간다. 어디서 F1이 꺾이는지 본다.

In [ ]:
for s in FIX:
    c = curve(Y, p, M, LABELS.index(s))
    print(f"[{s}]"); display(c.loc[[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]].round(3))

## 5. 정리
- B 절반에서 **재현율이 오르고 F1이 0.5일 때보다 크게 떨어지지 않으면** 그 기준값을 쓴다.
- 05 시연에서 쓰려면 `predict_dialogue(..., threshold=기준값 배열)`처럼 증상 순서(`config.LABELS`)대로 배열을 넘기면 된다.

In [ ]:
pick = "3개 수정, F1 최고(3개만)"   # B 절반 결과를 보고 고른 설정으로 바꾸기
final = pd.Series(settings[pick][1] if np.ndim(settings[pick][1]) else [0.5] * len(LABELS), index=LABELS, name="기준값")
final.to_csv(f"{OUT}/최종_기준값.csv", encoding="utf-8-sig")
print("고른 설정:", pick); display(final.to_frame())
print("05에 넣을 배열:", [float(x) for x in final.values])